# Titian — Provenance for the RDD API

The RDD API is Scala, so this notebook drives `spark-shell` from a bash cell, running
[`rdd_demo.scala`](rdd_demo.scala): a word-count-style aggregation whose corrupt
result is traced backward to the exact input line with `LineageContext` /
`getLineage` / `goBack` / `show` — the original Titian API, on stock Spark 4.

The script `require`s every assertion and prints `RDD-DEMO-ALL-OK` only if the full
capture-and-trace round trip succeeded.

In [ ]:
%%bash
set -e
export BIGASTERISK_HOME="${BIGASTERISK_HOME:-$(cd .. && pwd)}"
# A source checkout keeps the jars under modules/*/target; the Docker image under jars/.
if [ -z "$SPARK_JARS" ]; then
  SPARK_JARS="$(ls $BIGASTERISK_HOME/modules/*/target/scala-2.13/bigasterisk-*.jar 2>/dev/null \
                || ls $BIGASTERISK_HOME/jars/bigasterisk-*.jar)"
  SPARK_JARS="$(echo "$SPARK_JARS" | paste -sd, -)"
fi
FASTUTIL_JAR="${FASTUTIL_JAR:-$(ls $BIGASTERISK_HOME/jars/fastutil*.jar 2>/dev/null | head -1)}"
FASTUTIL_JAR="${FASTUTIL_JAR:-$(find ~/Library/Caches/Coursier ~/.cache/coursier -name 'fastutil-8.5.15.jar' 2>/dev/null | head -1)}"
if [ -z "$SPARK_HOME" ]; then
  SPARK_HOME="$(python3 -c 'import pyspark, os; print(os.path.dirname(pyspark.__file__))')"
fi

"$SPARK_HOME/bin/spark-shell" \
  --master 'local[2]' \
  --jars "$SPARK_JARS,$FASTUTIL_JAR" \
  --conf spark.ui.enabled=false \
  -i "$BIGASTERISK_HOME/notebooks/rdd_demo.scala" 2>/dev/null <<'SHELL_EOF' | tee /tmp/bigasterisk-rdd-demo.out | grep -E 'TOTALS|witness|TRACE|OK'
:quit
SHELL_EOF

grep -q 'RDD-DEMO-ALL-OK' /tmp/bigasterisk-rdd-demo.out

If the cell above ends with `RDD-DEMO-ALL-OK`, the RDD path captured lineage,
traced the corrupt aggregate backward to `electronics,99999`, and verified the
forward trace — all on stock Spark 4.